In [8]:
import numpy as np
from pathlib import Path

In [9]:
script_path = Path('./script.sh')

In [10]:
# DM 质量与对应的 sigma_n baseline
m_grid = np.array([1.0, 10.0, 100.0, 1.0e3, 1.0e4])
center_line = np.array([2.15e-25, 3.51e-27, 8.49e-28, 4.62e-27, 4.96e-26])

# 每个质量点相对于 baseline 的截面比例
log_cs_min = -4
log_cs_max = 0
n_cs = 5
cs_grid = np.logspace(log_cs_min, log_cs_max, n_cs)
n_m = m_grid.size

In [11]:
assert m_grid.shape == center_line.shape
assert np.all(np.diff(m_grid) > 0.0)
assert np.all(center_line > 0.0)

In [12]:
m_grid

array([1.e+00, 1.e+01, 1.e+02, 1.e+03, 1.e+04])

In [13]:
center_line

array([2.15e-25, 3.51e-27, 8.49e-28, 4.62e-27, 4.96e-26])

In [14]:
script_header = r'''#!/usr/bin/env bash
set -Eeuo pipefail

# Each parameter point runs as a one-process MPI singleton. The shell keeps
# MAX_JOBS independent parameter points running concurrently.
readonly SCRIPT_DIR="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd -P)"
readonly RUN_DIR="${SCRIPT_DIR}/DaMaSCUS-CRUST/bin"
readonly EXECUTABLE="${RUN_DIR}/DaMaSCUS-CRUST"
readonly CONFIG="${RUN_DIR}/config.cfg"
readonly MAX_JOBS="${MAX_JOBS:-16}"
readonly LOG_DIR="${LOG_DIR:-${SCRIPT_DIR}/logs}"
readonly TOTAL_TASKS=__TOTAL_TASKS__

if ! [[ "${MAX_JOBS}" =~ ^[1-9][0-9]*$ ]]; then
    printf 'ERROR: MAX_JOBS must be a positive integer (got %q).\n' "${MAX_JOBS}" >&2
    exit 2
fi
if [[ ! -x "${EXECUTABLE}" ]]; then
    printf 'ERROR: executable is missing or not executable: %s\n' "${EXECUTABLE}" >&2
    exit 2
fi
if [[ ! -r "${CONFIG}" ]]; then
    printf 'ERROR: config file is not readable: %s\n' "${CONFIG}" >&2
    exit 2
fi

mkdir -p -- "${LOG_DIR}"
cd -- "${RUN_DIR}"

declare -i submitted=0
declare -i active=0
declare -i failures=0
declare -a active_pids=()

run_task() {
    local task_id="$1"
    local mass="$2"
    local sigma="$3"
    local cross_section_type="$4"
    local v_min="$5"
    local log_file
    local status

    printf -v log_file '%s/task_%04d_mDM=%s_sigma_%s=%s_vMin=%s.log' "${LOG_DIR}" "${task_id}" "${mass}" "${cross_section_type}" "${sigma}" "${v_min}"
    printf '[%d/%d] START mDM=%s sigma_%s=%s vMin=%s\n' "${task_id}" "${TOTAL_TASKS}" "${mass}" "${cross_section_type}" "${sigma}" "${v_min}"

    if "${EXECUTABLE}" "${CONFIG}" "${mass}" "${sigma}" "${cross_section_type}" "${v_min}" >"${log_file}" 2>&1; then
        printf '[%d/%d] DONE  mDM=%s sigma_%s=%s vMin=%s\n' "${task_id}" "${TOTAL_TASKS}" "${mass}" "${cross_section_type}" "${sigma}" "${v_min}"
        return 0
    else
        status=$?
        printf '[%d/%d] FAIL  mDM=%s sigma_%s=%s vMin=%s (exit=%d, log=%s)\n' "${task_id}" "${TOTAL_TASKS}" "${mass}" "${cross_section_type}" "${sigma}" "${v_min}" "${status}" "${log_file}" >&2
        return "${status}"
    fi
}

reap_one() {
    local pid="${active_pids[0]}"
    if ! wait "${pid}"; then
        failures+=1
    fi
    active_pids=("${active_pids[@]:1}")
    active=${#active_pids[@]}
}

queue_task() {
    submitted+=1
    run_task "${submitted}" "$1" "$2" "$3" "$4" &
    active_pids+=("$!")
    active=${#active_pids[@]}
    if ((active >= MAX_JOBS)); then
        reap_one
    fi
}

stop_children() {
    trap - INT TERM
    local pid
    for pid in "${active_pids[@]}"; do
        kill "${pid}" 2>/dev/null || true
    done
    wait 2>/dev/null || true
    printf 'Interrupted; stopped active tasks.\n' >&2
    exit 130
}

trap stop_children INT TERM
'''

script_footer = r'''while ((active > 0)); do
    reap_one
done

if ((failures > 0)); then
    printf 'Finished %d tasks with %d failure(s). See logs in %s\n' "${submitted}" "${failures}" "${LOG_DIR}" >&2
    exit 1
fi

printf 'Finished all %d tasks successfully. Logs: %s\n' "${submitted}" "${LOG_DIR}"
'''

script_header = script_header.replace('__TOTAL_TASKS__', str(n_m * n_cs))
with script_path.open(mode='w', encoding='utf-8') as file:
    file.write(script_header)
    for i in range(n_m):
        m_dm = m_grid[i]
        for cs_scale in cs_grid:
            sigma_n = center_line[i] * cs_scale
            file.write(f"queue_task {m_dm:.12g} {sigma_n:.12g} n 0.0\n")
    file.write(script_footer)

script_path.chmod(0o755)
